# BeatAML Internal GNN Benchmark Fix

Notebook-first runner for the leakage-reduced internal benchmark: train/validate on BeatAML Waves1+2 and test on Waves3+4. The encoder excludes `treated_with` response edges; patient scaling is fit on the training patients only; the KTSP-compatible 20/20 and 10/10 drug filter is exported as a separate summary table.

## 1. Configuration

In [1]:
# Run flag: set False when you only want to inspect existing output tables.
RUN_INTERNAL_GNN = False

# Match the original internal notebook hyperparameters so the fix isolates leakage/split changes.
EPOCHS = 700
PATIENCE = 50
HIDDEN_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.3
HEADS = 4
LR = 0.0003
WEIGHT_DECAY = 0.0001
SCHEDULER_PATIENCE = 30
SCHEDULER_FACTOR = 0.5
MIN_LR = 0.000001
# Validation is drawn from Waves1+2 only; Waves3+4 remains the untouched internal test set.
VAL_FRACTION = 0.2
SEED = 42
BATCH_SIZE = 65536

KG_VARIANT = "beataml_primekg_pathway_disease_embeddings"
RUN_NAME = "primekg_pathway_disease_embeddings"
KG_EXPORT_DIR = f"data/interim/KG_Export/{KG_VARIANT}"

## 2. Setup

In [2]:
from pathlib import Path
import importlib
import os
import sys

import pandas as pd

candidate = Path.cwd().resolve()
repo_root = candidate.parent if candidate.name == "notebooks" else candidate
while not (repo_root / "AGENTS.md").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from scripts.pipeline.common import benchmark_utils
from scripts.pipeline.gnn_internal import train as train_gnn_internal

importlib.reload(benchmark_utils)
importlib.reload(train_gnn_internal)

TABLES = benchmark_utils.GNN_INTERNAL_TABLES / RUN_NAME
CHECKPOINTS = benchmark_utils.GNN_INTERNAL_CHECKPOINTS / RUN_NAME
LOGS = benchmark_utils.GNN_INTERNAL_LOGS / RUN_NAME

repo_root


WindowsPath('D:/School Documents/Lab/KG AML/Code')

## 3. Run Internal GNN

In [3]:
# Primary comparison metrics exported by the backend are AUROC and balanced_accuracy.
# Checkpointing uses validation macro per-drug AUROC; the threshold is selected on validation macro per-drug balanced accuracy.
train_args = [
    "--epochs", str(EPOCHS),
    "--patience", str(PATIENCE),
    "--hidden-dim", str(HIDDEN_DIM),
    "--num-layers", str(NUM_LAYERS),
    "--dropout", str(DROPOUT),
    "--heads", str(HEADS),
    "--lr", str(LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--scheduler-patience", str(SCHEDULER_PATIENCE),
    "--scheduler-factor", str(SCHEDULER_FACTOR),
    "--min-lr", str(MIN_LR),
    "--val-fraction", str(VAL_FRACTION),
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--kg-export-dir", str(KG_EXPORT_DIR),
    "--kg-variant", str(KG_VARIANT),
    "--run-name", str(RUN_NAME),
]

if RUN_INTERNAL_GNN:
    train_gnn_internal.main(train_args)


## 4. Inspect Summary

In [4]:
summary = pd.read_csv(TABLES / "summary.csv")

# Read these columns first: test_auroc, test_balanced_accuracy,
# ktsp_subset_macro_auroc, and ktsp_subset_macro_bacc.
summary


,kg_variant,run_name,split_strategy,selection_metric,best_epoch,best_val_macro_per_drug_auroc,best_val_auroc,best_val_macro_per_drug_bacc_at_0_5,threshold_selection_metric,threshold,...,test_ap,test_balanced_accuracy,test_f1,test_accuracy,macro_per_drug_auroc,macro_per_drug_bacc,weighted_per_drug_auroc,n_ktsp_eligible_drugs,ktsp_subset_macro_auroc,ktsp_subset_macro_bacc
0,beataml_primekg_pathway_disease_embeddings,primekg_pathway_disease_embeddings,patient_disjoint,val_macro_per_drug_auroc,77,0.704,0.8246,0.5542,val_macro_per_drug_bacc,0.39,...,0.4126,0.735,0.3091,0.5575,0.612,0.586,0.5754,37,0.5332,0.5104


In [5]:
per_drug = pd.read_csv(TABLES / "per_drug_metrics.csv")
metric_cols = ["drug", "n_test", "n_pos", "n_neg", "auroc", "balanced_accuracy"]

# Per-drug internal metrics on the full Waves3+4 test set.
per_drug[metric_cols].sort_values(["balanced_accuracy", "auroc"], ascending=False).head(20)


,drug,n_test,n_pos,n_neg,auroc,balanced_accuracy
122,SB-202190,13,1,12,0.916667,0.958333
47,GDC-0879,178,1,177,0.954802,0.954802
22,Bay 11-7085,158,3,155,0.776344,0.812903
105,PP2,13,1,12,0.916667,0.791667
134,Staurosporine,13,2,11,0.818182,0.772727
77,Lenalidomide,163,5,158,0.858228,0.738608
138,Tandutinib (MLN518),12,1,11,0.909091,0.727273
73,KW-2449,173,7,166,0.746127,0.726764
119,Roscovitine (CYC-202),178,2,176,0.599432,0.721591
151,XMD 8-87,53,1,52,0.615385,0.721154


## 5. KTSP-Compatible Drug Filter

In [6]:
ktsp_compatible = pd.read_csv(TABLES / "ktsp_compatible_metrics.csv")
ktsp_compatible["ktsp_eligible"].value_counts(dropna=False).rename_axis("ktsp_eligible").reset_index(name="n_drugs")

,ktsp_eligible,n_drugs
0,False,129
1,True,37


In [7]:
ktsp_subset = ktsp_compatible[ktsp_compatible["ktsp_eligible"]].copy()
ktsp_metric_cols = [
    "drug",
    "train_sensitive_auc100",
    "train_resistant_auc100",
    "test_sensitive_auc100",
    "test_resistant_auc100",
    "n_test",
    "auroc",
    "balanced_accuracy",
]

# KTSP-compatible internal subset: train has >=20/20 and test has >=10/10.
# Compare drugs by both AUROC and balanced_accuracy.
ktsp_subset[ktsp_metric_cols].sort_values(["balanced_accuracy", "auroc"], ascending=False).head(20)

,drug,train_sensitive_auc100,train_resistant_auc100,test_sensitive_auc100,test_resistant_auc100,n_test,auroc,balanced_accuracy
0,17-AAG (Tanespimycin),45,291,38,137,175,0.656934,0.611602
1,A-674563,64,280,44,136,180,0.656334,0.608289
40,Dovitinib (CHIR-258),57,285,35,141,176,0.670517,0.588349
123,RAF265 (CHIR-265),26,310,39,138,177,0.569491,0.551839
117,Pelitinib (EKB-569),38,304,36,139,175,0.544964,0.529277
150,Tivozanib (AV-951),53,283,31,141,172,0.633379,0.524823
71,KI20227,33,305,13,110,123,0.519580,0.522727
50,GDC-0941,36,285,46,114,160,0.507723,0.517544
156,Vargetef,29,310,24,152,176,0.570175,0.513158
31,Cabozantinib,41,293,30,149,179,0.661521,0.510403


## 6. Training Log and Threshold

In [8]:
training_log = pd.read_csv(LOGS / "training_log.csv")
threshold_scan = pd.read_csv(TABLES / "threshold_scan.csv")

display(training_log.tail(10))
threshold_scan.sort_values(["macro_per_drug_bacc", "distance_to_0_5"], ascending=[False, True]).head(10)


,epoch,train_loss,val_auroc,val_ap,val_balanced_accuracy_at_0_5,val_macro_per_drug_auroc,val_macro_per_drug_ap,val_macro_per_drug_bacc_at_0_5,val_macro_per_drug_n_valid_auc,lr
117,118,0.917247,0.834651,0.412587,0.751026,0.700291,0.218991,0.554768,109,0.00015
118,119,0.909356,0.834717,0.412723,0.750655,0.700001,0.218495,0.553048,109,0.00015
119,120,0.913546,0.835016,0.413106,0.749749,0.701013,0.221023,0.552408,109,0.00015
120,121,0.884884,0.835001,0.413251,0.748406,0.700942,0.221022,0.545600,109,0.00015
121,122,0.906842,0.835049,0.413516,0.749111,0.700495,0.220066,0.545935,109,0.00015
122,123,0.895224,0.835130,0.413689,0.749348,0.700370,0.220768,0.545680,109,0.00015
123,124,0.895279,0.835161,0.413703,0.748709,0.700540,0.218442,0.545212,109,0.00015
124,125,0.888026,0.835243,0.413847,0.748910,0.700304,0.218800,0.545015,109,0.00015
125,126,0.896529,0.835166,0.413801,0.748812,0.699429,0.215598,0.544746,109,0.00015
126,127,0.886293,0.835203,0.413768,0.748879,0.698711,0.215847,0.542766,109,0.00015


,threshold,macro_per_drug_bacc,balanced_accuracy,macro_per_drug_sensitivity,macro_per_drug_specificity,n_valid_drugs,distance_to_0_5
38,0.39,0.600194,0.600194,0.769738,0.489546,109,0.11
35,0.36,0.599801,0.599801,0.814342,0.440195,109,0.14
34,0.35,0.599646,0.599646,0.831817,0.423496,109,0.15
37,0.38,0.599456,0.599456,0.780900,0.474592,109,0.12
36,0.37,0.599102,0.599102,0.797807,0.457332,109,0.13
33,0.34,0.596957,0.596957,0.845120,0.404396,109,0.16
30,0.31,0.596937,0.596937,0.890876,0.357480,109,0.19
41,0.42,0.596065,0.596065,0.713289,0.534905,109,0.08
31,0.32,0.595091,0.595091,0.870365,0.375125,109,0.18
32,0.33,0.594703,0.594703,0.852306,0.392681,109,0.17


## 7. Statistical Significance Testing (10 Runs)
This section automates the 10-run protocol to calculate the P-Value using an independent T-test. Because it recalculates both the GNN and the XGB baseline 10 times with different random seeds, executing this cell might take a long time.

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
from scripts.pipeline.tabular_internal import run as tabular_run

RUN_SIGNIFICANCE = True # Set to True when you want to run the 10-seed experiment

if RUN_SIGNIFICANCE:
    seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
    
    gnn_aurocs, gnn_baccs = [], []
    xgb_aurocs, xgb_baccs = [], []
    
    for s in seeds:
        print(f"\n{'='*50}\nIteration for Seed: {s}\n{'='*50}")
        
        # --- 1. Run GNN ---
        gnn_args = train_args.copy()
        gnn_args[gnn_args.index("--seed") + 1] = str(s)
        gnn_args[gnn_args.index("--run-name") + 1] = f"{RUN_NAME}_significance_{s}"
        
        print(f"Training GNN with seed {s}...")
        train_gnn_internal.main(gnn_args)
        
        gnn_sum_path = benchmark_utils.GNN_INTERNAL_TABLES / f"{RUN_NAME}_significance_{s}" / "summary.csv"
        gnn_summary = pd.read_csv(gnn_sum_path)
        gnn_aurocs.append(gnn_summary['test_auroc'].values[0])
        gnn_baccs.append(gnn_summary['test_balanced_accuracy'].values[0])
        
        # --- 2. Run Tabular (XGBoost) ---
        print(f"Training Tabular (XGBoost) with seed {s}...")
        tabular_run.main(["--seed", str(s), "--xgb-iter", "20"])
        
        tab_summary = pd.read_csv(benchmark_utils.TABULAR_INTERNAL_TABLES / "summary.csv")
        xgb_row = tab_summary[tab_summary['model'] == 'xgb']
        xgb_aurocs.append(xgb_row['test_auroc'].values[0])
        xgb_baccs.append(xgb_row['test_balanced_accuracy'].values[0])

    # --- 3. Compute Statistical Significance ---
    print("\n" + "="*50)
    print("STATISTICAL SIGNIFICANCE RESULTS (10 Runs)")
    print("="*50)
    
    print(f"GNN AUROC: {np.mean(gnn_aurocs):.3f} +/- {np.std(gnn_aurocs):.3f}")
    print(f"XGB AUROC: {np.mean(xgb_aurocs):.3f} +/- {np.std(xgb_aurocs):.3f}\n")
    
    print(f"GNN BACC : {np.mean(gnn_baccs):.3f} +/- {np.std(gnn_baccs):.3f}")
    print(f"XGB BACC : {np.mean(xgb_baccs):.3f} +/- {np.std(xgb_baccs):.3f}\n")
    
    t_stat_auc, p_val_auc = ttest_ind(gnn_aurocs, xgb_aurocs)
    print(f"AUROC P-Value: {p_val_auc:.5f} -> {'Significant' if p_val_auc < 0.05 else 'Not Significant'}")
    
    t_stat_bacc, p_val_bacc = ttest_ind(gnn_baccs, xgb_baccs)
    print(f"BACC P-Value: {p_val_bacc:.5f} -> {'Significant' if p_val_bacc < 0.05 else 'Not Significant'}")
else:
    print("Skipping Significance Run. Set RUN_SIGNIFICANCE = True to execute.")